# Load packages

In [ ]:
#conda install -c bioconda gseapy

In [1]:
import gseapy as gp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from scipy.cluster.hierarchy import linkage, leaves_list

In [3]:
print(gp.__version__)

1.1.8


# Load data

# GSEA function

In [2]:
def run_gsea_analysis(rnk_file, gene_sets, outdir, fdr_threshold=0.05, permutation_num=1000, seed=42):
    """
    Run GSEA analysis on a ranked gene list file.
    
    Parameters:
    ----------
    rnk_file : str
        Path to the ranked gene list file
    gene_sets : str
        Path to the gene sets file (GMT format)
    outdir : str
        Output directory for GSEA results
    fdr_threshold : float
        FDR q-value threshold for significance
    permutation_num : int
        Number of permutations
    seed : int
        Random seed for reproducibility
        
    Returns:
    -------
    tuple
        (pre_res, gsea_res) - GSEA result object and processed dataframe
    """
    # Run GSEA prerank analysis
    pre_res = gp.prerank(
        rnk=rnk_file,
        gene_sets=gene_sets,
        outdir=outdir,
        permutation_num=permutation_num,
        seed=seed,
        no_plot=True,
    )
    
    # Process results
    gsea_res = pre_res.res2d
    gsea_res = gsea_res.sort_values("NES", ascending=True)
    
    # Classify results based on significance
    def classify(row):
        if row['FDR q-val'] < fdr_threshold:
            return 'up' if row['NES'] > 0 else 'down'
        else:
            return 'not_sig'
    
    gsea_res['significance'] = gsea_res.apply(classify, axis=1)
    
    return pre_res, gsea_res

# Run the function

## GCLC+VIM+ VS GCLC-VIM-

In [3]:
pre_res_1, gsea_res_1 = run_gsea_analysis(
    rnk_file='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/GCLC_VIM_--_GSEA.rnk',
    gene_sets='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/h.all.v2024.1.Hs.symbols.gmt',
    outdir='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/outputs',
    fdr_threshold=0.05,
    permutation_num=1000,
    seed=42
)

## GCLC+VIM+ VS GCLC-VIM+

In [4]:
pre_res_2, gsea_res_2 = run_gsea_analysis(
    rnk_file='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/GCLC_VIM_-+_GSEA.rnk',
    gene_sets='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/h.all.v2024.1.Hs.symbols.gmt',
    outdir='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/outputs',
    fdr_threshold=0.05,
    permutation_num=1000,
    seed=42
)

## GCLC+VIM+ VS GCLC+VIM-

In [5]:
pre_res_3, gsea_res_3 = run_gsea_analysis(
    rnk_file='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/GCLC_VIM_+-_GSEA.rnk',
    gene_sets='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/h.all.v2024.1.Hs.symbols.gmt',
    outdir='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/outputs',
    fdr_threshold=0.05,
    permutation_num=1000,
    seed=42
)

## Oxstress high VS low

In [6]:
pre_res_4, gsea_res_4 = run_gsea_analysis(
    rnk_file='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/Oxstress_group_GSEA.rnk',
    gene_sets='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/h.all.v2024.1.Hs.symbols.gmt',
    outdir='/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/outputs',
    fdr_threshold=0.05,
    permutation_num=1000,
    seed=42
)

# Plots

## Barplot

In [6]:
# ++ VS --
pre_res = pre_res_1
gsea_res = gsea_res_1

In [ ]:
# ++ VS -+
pre_res = pre_res_2
gsea_res = gsea_res_2

In [19]:
# ++ VS +-
pre_res = pre_res_3
gsea_res = gsea_res_3

In [7]:
terms = gsea_res['Term']
terms

5                HALLMARK_ESTROGEN_RESPONSE_LATE
8               HALLMARK_ESTROGEN_RESPONSE_EARLY
9               HALLMARK_CHOLESTEROL_HOMEOSTASIS
15                    HALLMARK_KRAS_SIGNALING_DN
17                 HALLMARK_BILE_ACID_METABOLISM
23                HALLMARK_FATTY_ACID_METABOLISM
24                           HALLMARK_PEROXISOME
26                    HALLMARK_ANDROGEN_RESPONSE
28                  HALLMARK_PANCREAS_BETA_CELLS
37              HALLMARK_PI3K_AKT_MTOR_SIGNALING
39                      HALLMARK_SPERMATOGENESIS
41            HALLMARK_OXIDATIVE_PHOSPHORYLATION
48            HALLMARK_UNFOLDED_PROTEIN_RESPONSE
49                           HALLMARK_DNA_REPAIR
47                         HALLMARK_ADIPOGENESIS
46                     HALLMARK_MTORC1_SIGNALING
45                    HALLMARK_PROTEIN_SECRETION
44                       HALLMARK_UV_RESPONSE_UP
43                      HALLMARK_MITOTIC_SPINDLE
42                           HALLMARK_GLYCOLYSIS
40                  

In [ ]:

#axs = pre_res.plot(terms=terms[0:3],
#                   #legend_kws={'loc': (1.2, 0)}, # set the legend loc
#                   show_ranking=True, # whether to show the second yaxis
#                   figsize=(3,4)
#                  )

selected_terms = pd.concat([terms[0:3], terms[-5: -1]])
axs = pre_res.plot(terms=selected_terms,
                   show_ranking=True,
                   figsize=(3, 4))
plt.tight_layout()
#plt.savefig("/Users/wenqchen/Desktop/Projects/Auria/Plots/scRNA/Oxstress_group_GSEA.pdf", format="pdf", bbox_inches='tight')


In [ ]:

palette = {'up': '#E69F00', 'down': '#0072B2', 'not_sig': 'lightgray'}

# plotting
plt.figure(figsize=(14, 14))
sns.barplot(
    data=gsea_res,
    x='NES',
    y='Term',
    hue='significance',
    dodge=False,
    palette=palette
)

plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('NES')
plt.ylabel('Pathway')
plt.title('Normalized enrichment scores')
plt.legend(
    title='significance',
    bbox_to_anchor=(1.01, 0.5),
    loc='center left',
    borderaxespad=0
)

plt.tight_layout()
#plt.savefig('/Users/wenqchen/Desktop/Projects/Auria/Plots/scRNA/Oxstress_group.pdf', format='pdf', bbox_inches='tight')
plt.show()

## Clustering bubble plot

In [8]:
gsea_res_1['Group'] = 'GCLC+VIM+_GCLC-VIM-'
gsea_res_2['Group'] = 'GCLC+VIM+_GCLC-VIM+'
gsea_res_3['Group'] = 'GCLC+VIM+_GCLC+VIM-'
gsea_res_4['Group'] = 'Oxstress_High_Oxstress_Low'

In [10]:
gsea_res_all = pd.concat([gsea_res_1, gsea_res_3, gsea_res_2, gsea_res_4])

In [11]:

plot_df = gsea_res_all.copy()

# Create bubble size: -log10(FDR q-val)
plot_df['-log10(p.adj)'] = -np.log10(plot_df['FDR q-val'].replace(0, 1e-4))  # avoid -inf

# Order terms by NES across all groups (use clustering if needed)
pivot = plot_df.pivot_table(index='Term', columns='Group', values='NES', fill_value=0)
linkage_result = linkage(pivot.values, method='average')
ordered_terms = pivot.index[leaves_list(linkage_result)]
plot_df['Term'] = pd.Categorical(plot_df['Term'], categories=ordered_terms, ordered=True)


/var/folders/rq/68pl8_s15_lc84mft22mqf012ws2td/T/ipykernel_70038/2278418764.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  plot_df['-log10(p.adj)'] = -np.log10(plot_df['FDR q-val'].replace(0, 1e-4))  # avoid -inf
/var/folders/rq/68pl8_s15_lc84mft22mqf012ws2td/T/ipykernel_70038/2278418764.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pivot = plot_df.pivot_table(index='Term', columns='Group', values='NES', fill_value=0)


In [12]:
# Remove "HALLMARK_" prefix from Term column for cleaner visualization
plot_df['Term'] = plot_df['Term'].str.replace('HALLMARK_', '')


In [13]:
plot_df

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,significance,Group,-log10(p.adj)
5,prerank,ESTROGEN_RESPONSE_LATE,-0.416066,-1.70617,0.0,0.022345,0.02,57/184,13.97%,TFF3;S100A9;TFF1;OLFM1;AGR2;SULT2B1;CHST8;HMGC...,down,GCLC+VIM+_GCLC-VIM-,1.650828
8,prerank,ESTROGEN_RESPONSE_EARLY,-0.392536,-1.613015,0.0,0.025696,0.046,48/188,11.49%,TFF3;TFF1;AR;KCNK15;MLPH;INHBB;OLFM1;SULT2B1;R...,down,GCLC+VIM+_GCLC-VIM-,1.590130
9,prerank,CHOLESTEROL_HOMEOSTASIS,-0.430442,-1.536725,0.013333,0.035379,0.094,38/70,22.78%,TM7SF2;SCD;FADS2;SREBF2;HMGCS1;FDPS;NSDHL;EBP;...,down,GCLC+VIM+_GCLC-VIM-,1.451255
15,prerank,KRAS_SIGNALING_DN,-0.350414,-1.34724,0.030888,0.134626,0.384,33/108,16.10%,TFAP2B;SPRR3;GPRC5C;SIDT1;KRT4;EDN1;EFHD1;CALC...,not_sig,GCLC+VIM+_GCLC-VIM-,0.870871
17,prerank,BILE_ACID_METABOLISM,-0.362164,-1.326825,0.041096,0.124459,0.439,28/86,16.45%,DIO1;AR;SULT2B1;DIO2;IDH2;ACSL1;PEX11A;GNMT;CY...,not_sig,GCLC+VIM+_GCLC-VIM-,0.904973
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,prerank,ESTROGEN_RESPONSE_EARLY,0.423245,1.846803,0.0,0.000577,0.002,69/185,17.75%,SFN;KRT8;KRT19;KRT18;CLDN7;MSMB;MUC1;KLK10;IL1...,up,Oxstress_High_Oxstress_Low,3.239015
7,prerank,XENOBIOTIC_METABOLISM,0.441237,1.870914,0.0,0.000721,0.002,72/158,24.91%,AKR1C2;NQO1;DDC;RBP4;HSD17B2;ABCC2;CYP27A1;PTG...,up,Oxstress_High_Oxstress_Low,3.142105
6,prerank,GLYCOLYSIS,0.4497,1.971298,0.0,0.0,0.0,92/180,28.01%,GPR87;ADORA2B;GPC1;CLDN3;ME1;DSC2;TFF3;ALDH7A1...,up,Oxstress_High_Oxstress_Low,4.000000
5,prerank,MYC_TARGETS_V1,0.467661,2.079079,0.0,0.0,0.0,126/193,41.40%,NME1;CDC45;CDC20;MCM7;POLD2;HDAC2;RUVBL2;MYC;C...,up,Oxstress_High_Oxstress_Low,4.000000


In [14]:
plot_df.to_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/scRNA/GSEA/Oxstress_group_GSEA.csv', index=False)

In [56]:
plot_df = plot_df[plot_df['-log10(p.adj)'] > 1.3]

In [ ]:


plt.figure(figsize=(10, 15))
bubble = sns.scatterplot(
    data=plot_df,
    x="Group",
    y="Term",
    size="-log10(p.adj)",
    hue="NES",
    palette="RdBu_r",  # Red = up, Blue = down
    sizes=(50, 300),
    edgecolor="gray",
    linewidth=0.5
)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.title("GSEA Bubble Plot by Group")
plt.tight_layout()

plt.xlabel("")
plt.xticks(rotation=90)
plt.ylabel("")

#plt.savefig('/Users/wenqchen/Desktop/Projects/Auria/Plots/scRNA/GCLCVIM_group_bubble.pdf', format='pdf', bbox_inches='tight')
plt.savefig('/Users/wenqchen/Desktop/Projects/Auria/Plots/scRNA/GCLCVIM_group_bubble_withoutfilter.pdf', format='pdf', bbox_inches='tight')
plt.show()
